# Setup

This version is configured to use **Ollama locally** for free.

**Before running the notebook:**
1. Install Ollama.
2. Run `ollama pull llama3.2` once in a terminal.
3. Start the Ollama server with `ollama serve` if it is not already running.


In [ ]:
!pip install -q 'crewai[litellm]'

In [ ]:
from crewai import Agent, Process, Crew, Task, LLM
from IPython.display import display, Markdown
import os

# Optional: keeps CrewAI from falling back to its default model in some setups
os.environ["MODEL"] = "ollama/llama3.2"


In [15]:
# connectivity check for Ollama (this should work if `ollama serve` is running locally)
import requests

try:
    response = requests.get("http://localhost:11434/api/tags", timeout=5)
    print("Ollama connection OK" if response.ok else f"Ollama responded with status {response.status_code}")
except Exception as e:
    print("Could not connect to Ollama at http://localhost:11434")
    print("Make sure Ollama is installed, `ollama pull llama3.2` has been run, and `ollama serve` is running.")
    print(f"Details: {e}")


Ollama connection OK


In [5]:
# Define the local LLM
llm = LLM(
    model="ollama/llama3.2",
    base_url="http://localhost:11434"
)


# Get the data

*   Interviewer
*   company
*   job position
*   job description



In [6]:
# Store the data
interviewer = input("Enter the name of the interviewer: ")
company = input("Enter the name of the company: ")
job_position = input("Enter the job position: ")
job_description = input("Enter the job description: ")

# Interviewer AI Agent

In [7]:
# Create the interver AI agent
interviewer_agent = Agent(
    role = f" You are {interviewer}, employed at {company}, and hiring for job position {job_position}",
    goal = f" you help the user user prepare for the job interview for job {job_position} with the following description: {job_description}",
    backstory = """
    Experience
    BEAT81
    Analytics Engineer

    BEAT81 · Full-timeBEAT81 · Full-time
    Apr 2024 - Present · 11 mos
    Berlin, Germany
    diconium
    Full-time · 4 yrs 2 mosFull-time · 4 yrs 2 mos

    Senior Cloud Architect
    Apr 2022 - Mar 2024 · 2 yrs
    Microsoft Azure, DBT and +4 skills

    Data Engineer
    Jan 2021 - Mar 2022 · 1 yr 3 mosJan 2021 to Mar 2022 · 1 yr 3 mos
    DBT, AWS and +4 skills

    Data Scientist
    Feb 2020 - Dec 2020 · 11 mos
    SQL and Python
    4flow logo

    Data Scientist
    4flow · Full-time4flow · Full-time
    Jun 2019 - Dec 2019 · 7 mosJun 2019 to Dec 2019 · 7 mos
    SQL and Python
    DCMN
    Full-time · 3 yrs 3 mos

    Data Scientist
    Apr 2017 - Apr 2019 · 2 yrs 1 moApr 2017 to Apr 2019 · 2 yrs 1 mo
    SQL and Python
    Data Analyst / Business Intelligence Manager

    Feb 2016 - Mar 2017 · 1 yr 2 mosFeb 2016 to Mar 2017 · 1 yr 2 mos
    SQL and Python
    iversity logo

    Data Analyst
    iversity · Full-timeiversity · Full-time
    Dec 2014 - Dec 2015 · 1 yr 1 moDec 2014 to Dec 2015 · 1 yr 1 mo
    SQL and R

    Education
    Technische Universität Berlin
    Master of Science - MS, Computational Neuroscience
    Sep 2009 - Aug 2013

    Universität Osnabrück
    Bachelor of Science - BS, Cognitive ScienceBachelor of Science - BS, Cognitive Science
    """,
    llm = llm
)

In [8]:
# Define the interview prep task
interview_prep_task = Task(
    description = f"""
    Interview the user for the job {job_position} with the following description: {job_description}
    """,
    expected_output = f"""
    ask only one question to the user that is relevant for the job {job_position} that has not been asked before
    """,
    agent = interviewer_agent,
    human_input = True
)

# AI Coach

In [9]:
# Build the AI interview coach agent
coach_agent = Agent(
    role = "Interview Coach",
    goal = f"I help the user prepare for the job interview for job {job_position} by grading the relevance of the answer",
    backstory = "You are an expert in job interviews",
    llm = llm)

In [10]:
# Define the Coaching Task
coaching_task = Task(
    description = """ Give feedback to the user on its answer """,
    expected_output = f""" Give feedback to the user on its answer with a bullet points on what was good, bad, and how to take it to the next level""",
    agent = coach_agent,
    context = [interview_prep_task]
)

# AI Crew

In [11]:
# Create an empty list for questions
interview_questions = []


In [13]:
crew = Crew(
    agents=[interviewer_agent, coach_agent],
    tasks=[interview_prep_task, coaching_task],
    verbose=True,
    process=Process.sequential,
    memory=False  # keep this off unless you also configure a non-OpenAI embedder
)
result = crew.kickoff()


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 27d32d45-6b14-49d3-9fdc-c438bfab2ab3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Interview the user for the job Data Analytics Working Student with the following description: About the    │
│  Role As a Working Student in our Analytics Team, you will work closely with our Tech, Product, Marketing and   │
│  Operations team to support our daily operations with analytical insights and help us to grow. You will be      │
│  joining a dynamic, fast-paced start-up where you’ll have ownership and impact from day one!  Your              │
│  Responsibilities Collaborate with stakeholders to understand data requirements  Support teams with data and    │
│  analytics requests and provide analytics solutions  Create and maintain company dashboards  Carry out          │
│  stand-alone analytics projects in marketing, operations or product  Assist in developing, improving and        │
│  scaling existing data pipelines  Your Profile Currently enrolled in a relevant study program with a strong     │
│  quantitative background (e.g. mathematics, computer science, data science, business informatics etc.)  Good    │
│  knowledge of Google Sheets  Good Knowledge of SQL  Good Knowledge of Python  Knowledge of common data          │
│  visualization tools (e.g. Tableau, Lookerstudio etc.)  Knowledge of dbt, GCP, BigQuery is a plus  Fluent in    │
│  English  OUR OFFER  WALK THE TALK: Unlimited BEAT81 workouts for you and one family member/friend: we          │
│  guarantee that you will get fit working here :)  YOUR DEVICE: Everyone has their own preferences when it       │
│  comes to technology - whether it’s Mac or Windows - you choose your laptop. Additionally, we provide Bose      │
│  noise-canceling headphones  HYBRID WORK: We have a beautiful office in the heart of Berlin                     │
│  (Weinmeisterstraße) with free drinks, healthy snacks and fruits. We like to spend time there together, but     │
│  you are also welcome to work from home  FLEXIBLE WORKING HOURS: Early bird or late riser? Shape your workday   │
│  to align with your personal commitments and preferences in coordination with your team and the business needs  │
│  TEAM RITUALS: Shared experiences build a strong team culture. That’s why we have our usuals such as regular    │
│  all hands meetings, Aperol Thursdays and joint workouts - there is something for everyone  TEAM EVENTS: Our    │
│  summer and Christmas parties have become as legendary as our annual off-sites and the perfect occasion to      │
│  spend quality time with team members                                                                           │
│                                                                                                                 │
│  ID: 8d0d1173-0677-498c-bb8c-1816a2cf0748                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent:  You are Matthias Glöel, employed at Beat81, and hiring for job position Data Analytics Working         │
│  Student                                                                                                        │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Interview the user for the job Data Analytics Working Student with the following description: About the    │
│  Role As a Working Student in our Analytics Team, you will work closely with our Tech, Product, Marketing and   │
│  Operations team to support our daily operations with analytical insights and help us to grow. You will be      │
│  joining a dynamic, fast-paced start-up where you’ll have ownership and impact from day one!  Your              │
│  Responsibilities Collaborate with stakeholders to understand data requirements  Support teams with data and    │
│  analytics requests and provide analytics solutions  Create and maintain company dashboards  Carry out          │
│  stand-alone analytics projects in marketing, operations or product  Assist in developing, improving and        │
│  scaling existing data pipelines  Your Profile Currently enrolled in a relevant study program with a strong     │
│  quantitative background (e.g. mathematics, computer science, data science, business informatics etc.)  Good    │
│  knowledge of Google Sheets  Good Knowledge of SQL  Good Knowledge of Python  Knowledge of common data          │
│  visualization tools (e.g. Tableau, Lookerstudio etc.)  Knowledge of dbt, GCP, BigQuery is a plus  Fluent in    │
│  English  OUR OFFER  WALK THE TALK: Unlimited BEAT81 workouts for you and one family member/friend: we          │
│  guarantee that you will get fit working here :)  YOUR DEVICE: Everyone has their own preferences when it       │
│  comes to technology - whether it’s Mac or Windows - you choose your laptop. Additionally, we provide Bose      │
│  noise-canceling headphones  HYBRID WORK: We have a beautiful office in the heart of Berlin                     │
│  (Weinmeisterstraße) with free drinks, healthy snacks and fruits. We like to spend time there together, but     │
│  you are also welcome to work from home  FLEXIBLE WORKING HOURS: Early bird or late riser? Shape your workday   │
│  to align with your personal commitments and preferences in coordination with your team and the business needs  │
│  TEAM RITUALS: Shared experiences build a strong team culture. That’s why we have our usuals such as regular    │
│  all hands meetings, Aperol Thursdays and joint workouts - there is something for everyone  TEAM EVENTS: Our    │
│  summer and Christmas parties have become as legendary as our annual off-sites and the perfect occasion to      │
│  spend quality time with team members                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent:  You are Matthias Glöel, employed at Beat81, and hiring for job position Data Analytics Working         │
│  Student                                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Current Task:                                                                                                  │
│                                                                                                                 │
│  Interview the user for the job Data Analytics Working Student with the following description:                  │
│                                                                                                                 │
│  About the Role As a Working Student in our Analytics Team, you will work closely with our Tech, Product,       │
│  Marketing and Operations team to support our daily operations with analytical insights and help us to grow.    │
│  You will be joining a dynamic, fast-paced start-up where you’ll have ownership and impact from day one!  Your  │
│  Responsibilities Collaborate with stakeholders to understand data requirements  Support teams with data and    │
│  analytics requests and provide analytics solutions  Create and maintain company dashboards  Carry out          │
│  stand-alone analytics projects in marketing, operations or product  Assist in developing, improving and        │
│  scaling existing data pipelines  Your Profile Currently enrolled in a relevant study program with a strong     │
│  quantitative background (e.g. mathematics, computer science, data science, business informatics etc.)  Good    │
│  knowledge of Google Sheets  Good Knowledge of SQL  Good Knowledge of Python  Knowledge of common data          │
│  visualization tools (e.g. Tableau, Lookerstudio etc.)  Knowledge of dbt, GCP, BigQuery is a plus  Fluent in    │
│  English                                                                                                        │
│                                                                                                                 │
│  OUR OFFER                                                                                                      │
│  WALK THE TALK: Unlimited BEAT81 workouts for you and one family member/friend: we guarantee that you will get  │
│  fit working here :)                                                                                            │
│  YOUR DEVICE: Everyone has their own preferences when it comes to technology - whether it’s Mac or Windows -    │
│  you choose your laptop. Additionally, we provide Bose noise-canceling headphones                               │
│   HYBRID WORK: We have a beautiful office in the heart of Berlin (Weinmeisterstraße) with free drinks, healthy  │
│  snacks and fruits. We like to spend time there together, but you are also welcome to work from home            │
│   FLEXIBLE WORKING HOURS: Early bird or late riser? Shape your workday to align with your personal commitments  │
│  and preferences in coordination with your team and the business needs                                          │
│   TEAM RITUALS: Shared experiences build a strong team culture. That’s why we have our usuals such as regular   │
│  all hands meetings, Aperol Thursdays and joint workouts - there is something for everyone                      │
│   TEAM EVENTS: Our summer and Christmas parties have become as legendary as our annual off-sites and the        │
│  perfect occasion to spend quality time with team membe

╭────────────────────────────────────────── 💬 Human Feedback Required ───────────────────────────────────────────╮
│                                                                                                                 │
│  Provide feedback on the Final Result above.                                                                    │
│                                                                                                                 │
│  • If you are happy with the result, simply hit Enter without typing anything.                                  │
│  • Otherwise, provide specific improvement requests.                                                            │
│  • You can provide multiple rounds of feedback until satisfied.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Interview Coach                                                                                         │
│                                                                                                                 │
│  Task:  Give feedback to the user on its answer                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Interview Coach                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Feedback on your answer**                                                                                    │
│                                                                                                                 │
│  Here is the feedback on your answer:                                                                           │
│                                                                                                                 │
│  * **Good points:**                                                                                             │
│          + You showed an understanding of the role and its responsibilities.                                    │
│          + You mentioned some relevant skills and tools that are required for the job (e.g. Google Sheets,      │
│  SQL, Python, data visualization tools).                                                                        │
│          + You demonstrated enthusiasm and interest in the company and its culture.                             │
│  * **Bad points:**                                                                                              │
│          + Your answer was quite general and lacking in specific examples.                                      │
│          + You didn't directly address potential challenges that a Data Analytics Working Student might face.   │
│          + You could have provided more insights on how you would overcome these challenges.                    │
│  * **How to take it to the next level:**                                                                        │
│          + **Specifically identify potential challenges:** Think about common pain points or difficulties that  │
│  you, as a future Data Analytics Working Student, might encounter. For example:                                 │
│                  - "I think one challenge I might face is ensuring data quality and consistency across          │
│  different datasets."                                                                                           │
│                  - "Another potential challenge could be meeting tight deadlines for analytics projects while   │
│  maintaining project clarity and user feedback."                                                                │
│          + **Show how you would overcome these challenges:** Explain the steps you would take to address these  │
│  challenges:                                                                                                    │
│                  - "To ensure data quality and consistency, I would prioritize testing and validating my        │
│  datasets before implementing changes. I would also work closely with our stakeholders to understand their      │
│  requirements and ensure that our tools and processes align with theirs."                                       │
│                  - "To meet tight deadlines while maintaining project clarity and user feedback, I would        │
│  create templates for analytics projects that include clear documentation of data sources and analytical        │
│  methodologies. I would also make time for regular check-ins with team members and stakeholders to gather       │
│  feedback and adjust the project scope as needed."                                                              │
│          + **Provide more insights on how you learned a

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:  Give feedback to the user on its answer                                                                 │
│  Agent: Interview Coach                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 27d32d45-6b14-49d3-9fdc-c438bfab2ab3                                                                       │
│  Final Output: **Feedback on your answer**                                                                      │
│                                                                                                                 │
│  Here is the feedback on your answer:                                                                           │
│                                                                                                                 │
│  * **Good points:**                                                                                             │
│          + You showed an understanding of the role and its responsibilities.                                    │
│          + You mentioned some relevant skills and tools that are required for the job (e.g. Google Sheets,      │
│  SQL, Python, data visualization tools).                                                                        │
│          + You demonstrated enthusiasm and interest in the company and its culture.                             │
│  * **Bad points:**                                                                                              │
│          + Your answer was quite general and lacking in specific examples.                                      │
│          + You didn't directly address potential challenges that a Data Analytics Working Student might face.   │
│          + You could have provided more insights on how you would overcome these challenges.                    │
│  * **How to take it to the next level:**                                                                        │
│          + **Specifically identify potential challenges:** Think about common pain points or difficulties that  │
│  you, as a future Data Analytics Working Student, might encounter. For example:                                 │
│                  - "I think one challenge I might face is ensuring data quality and consistency across          │
│  different datasets."                                                                                           │
│                  - "Another potential challenge could be meeting tight deadlines for analytics projects while   │
│  maintaining project clarity and user feedback."                                                                │
│          + **Show how you would overcome these challenges:** Explain the steps you would take to address these  │
│  challenges:                                                                                                    │
│                  - "To ensure data quality and consistency, I would prioritize testing and validating my        │
│  datasets before implementing changes. I would also work closely with our stakeholders to understand their      │
│  requirements and ensure that our tools and processes align with theirs."                                       │
│                  - "To meet tight deadlines while maintaining project clarity and user feedback, I would        │
│  create templates for analytics projects that include clear documentation of data sources and analytical        │
│  methodologies. I would also make time for regular check-ins with team members and stakeholders to gather       │
│  feedback and adjust the project scope as needed."                                                              │
│          + **Provide more insights on how you learned 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [14]:
# Check output
display(Markdown(result.tasks_output[0].raw))

Current Task: 

Interview the user for the job Data Analytics Working Student with the following description:

About the Role As a Working Student in our Analytics Team, you will work closely with our Tech, Product, Marketing and Operations team to support our daily operations with analytical insights and help us to grow. You will be joining a dynamic, fast-paced start-up where you’ll have ownership and impact from day one!  Your Responsibilities Collaborate with stakeholders to understand data requirements  Support teams with data and analytics requests and provide analytics solutions  Create and maintain company dashboards  Carry out stand-alone analytics projects in marketing, operations or product  Assist in developing, improving and scaling existing data pipelines  Your Profile Currently enrolled in a relevant study program with a strong quantitative background (e.g. mathematics, computer science, data science, business informatics etc.)  Good knowledge of Google Sheets  Good Knowledge of SQL  Good Knowledge of Python  Knowledge of common data visualization tools (e.g. Tableau, Lookerstudio etc.)  Knowledge of dbt, GCP, BigQuery is a plus  Fluent in English  

OUR OFFER  
WALK THE TALK: Unlimited BEAT81 workouts for you and one family member/friend: we guarantee that you will get fit working here :)  
YOUR DEVICE: Everyone has their own preferences when it comes to technology - whether it’s Mac or Windows - you choose your laptop. Additionally, we provide Bose noise-canceling headphones  
 HYBRID WORK: We have a beautiful office in the heart of Berlin (Weinmeisterstraße) with free drinks, healthy snacks and fruits. We like to spend time there together, but you are also welcome to work from home  
 FLEXIBLE WORKING HOURS: Early bird or late riser? Shape your workday to align with your personal commitments and preferences in coordination with your team and the business needs  
 TEAM RITUALS: Shared experiences build a strong team culture. That’s why we have our usuals such as regular all hands meetings, Aperol Thursdays and joint workouts - there is something for everyone  
 TEAM EVENTS: Our summer and Christmas parties have become as legendary as our annual off-sites and the perfect occasion to spend quality time with team members  

What do you think are some potential challenges that a Data Analytics Working Student in our Team might face, and how would you overcome them?